# BCAD Validation Error Viewer

Inspect the validation error distributions used by BCAD. The signed error is `y_true_val - y_pred_val`; the absolute error is `abs(y_true_val - y_pred_val)`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())

def load_metadata(run_id):
    with open(EXPERIMENTS_DIR / run_id / "metadata.json", encoding="utf-8") as f:
        return json.load(f)

def load_validation_predictions(run_id):
    metadata = load_metadata(run_id)
    paths = metadata.get("paths", {})
    rel_path = paths.get("predictions") if isinstance(paths, dict) else None
    path = PROJECT_ROOT / rel_path if rel_path else EXPERIMENTS_DIR / run_id / "healthy_val_predictions.csv"
    if not path.exists():
        raise FileNotFoundError(f"Validation prediction file not found: {path}")
    return pd.read_csv(path)

def target_from_error_col(col):
    return col.split("::", 1)[1] if "::" in col else col

def list_targets(run_id):
    pred = load_validation_predictions(run_id)
    return [target_from_error_col(col) for col in pred.columns if col.startswith("error::")]

def build_error_summary(pred, targets):
    rows = []
    for target in targets:
        values = pd.to_numeric(pred[f"error::{target}"], errors="coerce").dropna()
        abs_values = values.abs()
        rows.append({
            "target": target,
            "samples": len(values),
            "signed_mean": values.mean(),
            "signed_std": values.std(ddof=0),
            "signed_median": values.median(),
            "signed_skewness": values.skew(),
            "abs_mean": abs_values.mean(),
            "abs_std": abs_values.std(ddof=0),
            "abs_median": abs_values.median(),
            "abs_skewness": abs_values.skew(),
        })
    return pd.DataFrame(rows)

def normal_pdf(x, center, scale):
    return np.exp(-0.5 * ((x - center) / scale) ** 2) / (scale * np.sqrt(2 * np.pi))

def folded_normal_pdf(x, center, scale):
    return normal_pdf(x, center, scale) + normal_pdf(-x, center, scale)

def plot_bcad_validation_error_distribution(run_id, target="ALL", bins=120, variance_multiplier=5.0):
    pred = load_validation_predictions(run_id)
    all_targets = [target_from_error_col(col) for col in pred.columns if col.startswith("error::")]
    targets = all_targets if target == "ALL" else [target]
    if not targets:
        raise ValueError("No validation error columns found.")

    display(Markdown("### BCAD validation error summary"))
    summary = build_error_summary(pred, targets)
    display(summary.style.format({
        "signed_mean": "{:.4f}",
        "signed_std": "{:.4f}",
        "signed_median": "{:.4f}",
        "signed_skewness": "{:.3f}",
        "abs_mean": "{:.4f}",
        "abs_std": "{:.4f}",
        "abs_median": "{:.4f}",
        "abs_skewness": "{:.3f}",
    }))

    fig, axes = plt.subplots(len(targets), 2, figsize=(14, 3.4 * len(targets)), constrained_layout=True)
    if len(targets) == 1:
        axes = [axes]

    for row_axes, current_target in zip(axes, targets):
        signed_values = pd.to_numeric(pred[f"error::{current_target}"], errors="coerce").dropna()
        abs_values = signed_values.abs()

        ax_signed, ax_abs = row_axes
        ax_signed.hist(signed_values, bins=bins, color="#2f6fbb", alpha=0.78)
        ax_signed.axvline(0, color="black", linewidth=1, linestyle="--", label="zero")
        ax_signed.axvline(signed_values.mean(), color="#d62728", linewidth=1.5, label="mean")
        ax_signed.set_title(f"{current_target} | signed validation error")
        ax_signed.set_xlabel("error = y_true_val - y_pred_val")
        ax_signed.set_ylabel("count")
        ax_signed.legend()

        ax_abs.hist(abs_values, bins=bins, density=True, color="#f28e2b", alpha=0.55, label="abs validation error")
        x_max = max(abs_values.quantile(0.999), abs_values.max() * 0.6)
        x = np.linspace(0, x_max, 500)
        healthy_pdf = folded_normal_pdf(x, signed_values.mean(), signed_values.std(ddof=0))
        abnormal_pdf = normal_pdf(x, abs_values.mean(), abs_values.std(ddof=0) * np.sqrt(variance_multiplier))
        ax_abs.plot(x, healthy_pdf, color="#2f6fbb", linewidth=2, label="H: folded normal")
        ax_abs.plot(x, abnormal_pdf, color="#d62728", linewidth=2, label="A: Gaussian")
        ax_abs.axvline(abs_values.mean(), color="#d62728", linewidth=1.2, linestyle="--", label="A center")
        ax_abs.set_title(f"{current_target} | absolute validation error")
        ax_abs.set_xlabel("absolute error = abs(y_true_val - y_pred_val)")
        ax_abs.set_ylabel("count")
        ax_abs.legend()

    plt.show()

def save_bcad_validation_error_distribution(run_id, target="ALL", bins=120, variance_multiplier=5.0):
    figures_dir = PROJECT_ROOT / "results" / "kelmarsh" / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    pred = load_validation_predictions(run_id)
    all_targets = [target_from_error_col(col) for col in pred.columns if col.startswith("error::")]
    targets = all_targets if target == "ALL" else [target]
    fig, axes = plt.subplots(len(targets), 2, figsize=(14, 3.4 * len(targets)), constrained_layout=True)
    if len(targets) == 1:
        axes = [axes]
    for row_axes, current_target in zip(axes, targets):
        signed_values = pd.to_numeric(pred[f"error::{current_target}"], errors="coerce").dropna()
        abs_values = signed_values.abs()
        ax_signed, ax_abs = row_axes
        ax_signed.hist(signed_values, bins=bins, color="#2f6fbb", alpha=0.78)
        ax_signed.axvline(0, color="black", linewidth=1, linestyle="--", label="zero")
        ax_signed.axvline(signed_values.mean(), color="#d62728", linewidth=1.5, label="mean")
        ax_signed.set_title(f"{current_target} | signed validation error")
        ax_signed.set_xlabel("error = y_true_val - y_pred_val")
        ax_signed.set_ylabel("count")
        ax_signed.legend()
        ax_abs.hist(abs_values, bins=bins, density=True, color="#f28e2b", alpha=0.55, label="abs validation error")
        x_max = max(abs_values.quantile(0.999), abs_values.max() * 0.6)
        x = np.linspace(0, x_max, 500)
        healthy_pdf = folded_normal_pdf(x, signed_values.mean(), signed_values.std(ddof=0))
        abnormal_pdf = normal_pdf(x, abs_values.mean(), abs_values.std(ddof=0) * np.sqrt(variance_multiplier))
        ax_abs.plot(x, healthy_pdf, color="#2f6fbb", linewidth=2, label="H: folded normal")
        ax_abs.plot(x, abnormal_pdf, color="#d62728", linewidth=2, label="A: Gaussian")
        ax_abs.axvline(abs_values.mean(), color="#d62728", linewidth=1.2, linestyle="--", label="A center")
        ax_abs.set_title(f"{current_target} | absolute validation error")
        ax_abs.set_xlabel("absolute error = abs(y_true_val - y_pred_val)")
        ax_abs.set_ylabel("count")
        ax_abs.legend()
    suffix = "all_targets" if target == "ALL" else target.replace("/", "_").replace(" ", "_")
    vm_label = f"vm{variance_multiplier:g}".replace(".", "p")
    out_path = figures_dir / f"bcad_validation_error_distribution_{run_id}_{suffix}_{vm_label}.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path

def load_bcad_window_trend(run_id, q=0.99, variance_multiplier=5.0, min_consecutive=6):
    rows = []
    run_dir = EXPERIMENTS_DIR / run_id
    vm_label = f"{variance_multiplier:g}".replace(".", "p")
    pattern = f"bcad_performance_abs_w*_q{int(q * 1000):03d}_vm{vm_label}_c{min_consecutive}.json"
    for path in sorted(run_dir.glob(pattern)):
        setting = path.stem.replace("bcad_performance_", "")
        try:
            w = int(setting.split("_w", 1)[1].split("_", 1)[0])
        except (IndexError, ValueError):
            continue
        perf = json.loads(path.read_text(encoding="utf-8"))
        event = perf["event_level"]
        alarm = perf["alarm_episode_level"]
        true_ep = alarm["true_alarm_episodes"]
        op_fa_ep = alarm["operational_false_alarm_episodes"]
        precision = true_ep / (true_ep + op_fa_ep) if (true_ep + op_fa_ep) else 0
        recall = event["detection_rate"]
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        rows.append({
            "window_size": w,
            "hours": w / 6,
            "detected_events": event["detected_events"],
            "total_events": event["total_events"],
            "miss_rate": event["miss_rate"],
            "operational_false_alarm_rate": alarm["operational_false_alarm_rate"],
            "total_alarm_episodes": alarm["total_alarm_episodes"],
            "true_alarm_episodes": true_ep,
            "operational_false_alarm_episodes": op_fa_ep,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "setting": setting,
        })
    if not rows:
        raise FileNotFoundError(f"No BCAD window trend files found with pattern: {pattern}")
    return pd.DataFrame(rows).sort_values("window_size").reset_index(drop=True)

def plot_bcad_window_trend(run_id, q=0.99, variance_multiplier=5.0, min_consecutive=6):
    trend = load_bcad_window_trend(run_id, q, variance_multiplier, min_consecutive)
    display(Markdown("### BCAD window-size trend"))
    display(trend[[
        "window_size", "hours", "detected_events", "total_events", "operational_false_alarm_rate",
        "precision", "recall", "f1_score", "total_alarm_episodes", "operational_false_alarm_episodes",
    ]].style.format({
        "hours": "{:.1f}",
        "operational_false_alarm_rate": "{:.2%}",
        "precision": "{:.2%}",
        "recall": "{:.2%}",
        "f1_score": "{:.3f}",
    }))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    axes[0].plot(trend["window_size"], trend["f1_score"], marker="o", color="#1f77b4", label="F1 score")
    axes[0].plot(trend["window_size"], trend["precision"], marker="o", color="#2ca02c", label="Precision")
    axes[0].plot(trend["window_size"], trend["recall"], marker="o", color="#ff7f0e", label="Recall")
    axes[0].set_ylabel("score")
    axes[0].set_ylim(0, 1.02)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[0].set_title("BCAD performance vs window size")

    axes[1].plot(trend["window_size"], trend["operational_false_alarm_rate"], marker="o", color="#d62728", label="Operational FA rate")
    axes[1].plot(trend["window_size"], trend["miss_rate"], marker="o", color="#9467bd", label="Miss rate")
    axes[1].set_xlabel("window size (10-min points)")
    axes[1].set_ylabel("rate")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.show()
    return trend


In [ ]:
runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment metadata found in {EXPERIMENTS_DIR}")

default_run = runs[-1]

if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=runs, value=default_run, description="Run")
    target_dropdown = widgets.Dropdown(options=["ALL"] + list_targets(default_run), value="ALL", description="Target")
    bins_slider = widgets.IntSlider(value=120, min=30, max=250, step=10, description="Bins")
    vm_slider = widgets.FloatSlider(value=5.0, min=1.5, max=12.0, step=0.5, description="VM")
    show_button = widgets.Button(description="Show BCAD distribution", button_style="primary")
    save_button = widgets.Button(description="Save figure")
    trend_button = widgets.Button(description="Plot W trend")
    output = widgets.Output()

    def refresh_targets(*_):
        targets = ["ALL"] + list_targets(run_dropdown.value)
        target_dropdown.options = targets
        target_dropdown.value = "ALL"

    def on_show(_):
        with output:
            clear_output(wait=True)
            plot_bcad_validation_error_distribution(run_dropdown.value, target_dropdown.value, bins_slider.value, vm_slider.value)

    def on_save(_):
        with output:
            clear_output(wait=True)
            path = save_bcad_validation_error_distribution(run_dropdown.value, target_dropdown.value, bins_slider.value, vm_slider.value)
            display(Markdown(f"Saved figure: `{path}`"))
            plot_bcad_validation_error_distribution(run_dropdown.value, target_dropdown.value, bins_slider.value, vm_slider.value)

    def on_trend(_):
        with output:
            clear_output(wait=True)
            plot_bcad_window_trend(run_dropdown.value, q=0.99, variance_multiplier=5.0, min_consecutive=6)

    run_dropdown.observe(refresh_targets, names="value")
    show_button.on_click(on_show)
    save_button.on_click(on_save)
    trend_button.on_click(on_trend)

    display(widgets.VBox([
        widgets.HBox([run_dropdown, target_dropdown, bins_slider, vm_slider]),
        widgets.HBox([show_button, save_button, trend_button]),
        output,
    ]))
else:
    print("ipywidgets is not available. Use plot_bcad_validation_error_distribution(...) and plot_bcad_window_trend(...) manually.")
